In [2]:
import pandas as pd
import csv

# ========== 全局去重配置 ==========
DEDUP_KEEP = 'first'   # 'first' 或 'last'

# ========== 标签合并函数 ==========
def combine_labels_by_any(df, label_columns):
    """多列二值标签（0/1）：只要任意一列为1，结果即为1，否则0"""
    existing = [col for col in label_columns if col in df.columns]
    if not existing:
        return pd.Series([''] * len(df))
    combined = df[existing].max(axis=1).fillna(0).astype(int)
    return combined

# ========== 文件配置列表 ==========
file_configs = [
    # {
    #     "path": "es_hf_102024.csv",
    #     "sep": ",",
    #     "tweet_column": "text",
    #     "label_config": {
    #         "columns": [
    #             "Hate_Speech",
    #             "Targeted_Harassment",
    #             "NSFW_Sexual",
    #             "Violence_Incitement",
    #             "Dangerous_Ideology",
    #             "Profanity_Slang"
    #         ],
    #         "func": combine_labels_by_any   # 使用默认合并函数
    #     },
    #     "header": 0
    # },
    # 其他文件示例（单列标签）
    {
        "path": "es_hf_102024.csv",
        "sep": ",",
        "tweet_column": "text",
        "label_config": "labels",   # 直接写列名
        "header": 0
    },
    {
        "path": "es.csv",
        "sep": ",",
        "tweet_column": "text",
        "label_config": "label",   # 直接写列名
        "header": 0
    },
    # {
    #     "path": "in_hf.csv",
    #     "sep": ",",
    #     "tweet_column": "text",
    #     "label_config": "labels",   # 直接写列名
    #     "header": 0
    # },
    
]

# ========== 读取与提取函数 ==========
def read_file_with_fallback(file_config):
    file_path = file_config["path"]
    sep = file_config.get("sep", ",")
    header = file_config.get("header", 0)
    encoding_list = file_config.get("encoding_try", ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252'])
    for enc in encoding_list:
        try:
            df = pd.read_csv(file_path, sep=sep, header=header, encoding=enc)
            print(f"  ✓ 使用编码 {enc} 成功读取 {file_path}")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"  ✗ 使用编码 {enc} 读取失败: {e}")
            continue
    print(f"  ⚠ 所有编码均失败，使用 latin-1 强制读取")
    return pd.read_csv(file_path, sep=sep, header=header, encoding='latin-1')

def extract_tweets_and_labels(df, tweet_column, label_config):
    """提取推文和标签，支持单列字符串或多列字典配置"""
    if tweet_column not in df.columns:
        print(f"  ✗ 未找到推文列 '{tweet_column}'，实际列名: {df.columns.tolist()}")
        return [], []
    
    # 删除推文为空的记录
    non_empty = df[tweet_column].notna()
    valid_df = df[non_empty].copy()
    tweets = valid_df[tweet_column].tolist()

    # 处理标签
    if label_config is None:
        labels = [''] * len(tweets)
    elif isinstance(label_config, str):
        # 旧式：单一列名
        if label_config not in df.columns:
            print(f"  ⚠ 未找到标签列 '{label_config}'，全部填充为空")
            labels = [''] * len(tweets)
        else:
            labels = valid_df[label_config].fillna('').astype(str).tolist()
    elif isinstance(label_config, dict):
        cols = label_config.get('columns', [])
        func = label_config.get('func', combine_labels_by_any)
        # 在原始 df 上计算标签（避免切片后索引失效）
        combined = func(df, cols)
        labels = combined[non_empty].fillna('').astype(str).tolist()
    else:
        raise TypeError(f"label_config 类型错误: {type(label_config)}")

    print(f"  提取到 {len(tweets)} 条推文，有效标签数: {len([l for l in labels if l != ''])}")
    return tweets, labels

def main():
    all_tweets = []
    all_labels = []

    for idx, config in enumerate(file_configs, 1):
        print(f"\n处理文件 {idx}: {config['path']}")
        df = read_file_with_fallback(config)
        tweets, labels = extract_tweets_and_labels(
            df,
            config["tweet_column"],
            config.get("label_config")  # 注意这里用的是 label_config
        )
        all_tweets.extend(tweets)
        all_labels.extend(labels)

    # 保存原始合并数据
    raw_df = pd.DataFrame({'tweet': all_tweets, 'label': all_labels})
    raw_output = "all_datasets_with_labels.csv"
    raw_df.to_csv(raw_output, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
    print(f"\n✓ 原始合并: {len(raw_df)} 条记录，已保存至 {raw_output}")

    # 去重（基于 tweet）
    dedup_df = raw_df.drop_duplicates(subset=['tweet'], keep=DEDUP_KEEP)
    removed = len(raw_df) - len(dedup_df)
    print(f"✓ 去重后: {len(dedup_df)} 条记录（移除 {removed} 条重复）")

    dedup_output = "all_datasets_with_labels_dedup.csv"
    dedup_df.to_csv(dedup_output, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
    print(f"✓ 去重结果已保存至 {dedup_output}")
    print("\n去重后前5条预览：")
    print(dedup_df.head())

if __name__ == "__main__":
    main()


处理文件 1: es_hf_102024.csv
  ✓ 使用编码 utf-8 成功读取 es_hf_102024.csv
  提取到 29855 条推文，有效标签数: 29855

处理文件 2: es.csv
  ✓ 使用编码 utf-8 成功读取 es.csv
  提取到 24110 条推文，有效标签数: 24110

✓ 原始合并: 53965 条记录，已保存至 all_datasets_with_labels.csv
✓ 去重后: 50787 条记录（移除 3178 条重复）
✓ 去重结果已保存至 all_datasets_with_labels_dedup.csv

去重后前5条预览：
                                               tweet label
0  Eran tan pero tan feministas que invisibilizab...   0.0
1  @USER @USER @USER Me carga en lo q se convirti...   0.0
2  , ¿Sabrán las femiorcas como @USER y todo el f...   1.0
3  @USER @USER @USER @USER Una vecina que nada te...   0.0
4   @USER Debajo de que piedra estaba ese flaiterio?   0.0


In [3]:
import pandas as pd

# 读取 parquet 文件
df = pd.read_parquet("es-00000-of-00001.parquet", engine="pyarrow")  # 或 engine="fastparquet"

# 保存为 CSV
df.to_csv("es-00000-of-00001.csv", index=False)

print("转换完成，已保存为 es-00000-of-00001.csv")

转换完成，已保存为 es-00000-of-00001.csv


In [ ]:
import pandas as pd
import csv

# ========== 去重配置 ==========
DEDUP_KEEP = 'first'   # 'first' 或 'last'

# ========== 文件配置列表 ==========
file_configs = [
    # 示例1：多列二值合并（只要有一个1就是有毒）
    {
        "path": "English.csv",
        "sep": ",",
        "tweet_column": "text",
        "toxicity_columns": [
            "Hate_Speech",
            "Targeted_Harassment",
            "NSFW_Sexual",
            "Violence_Incitement",
            "Dangerous_Ideology",
            "Profanity_Slang"
        ],
        "header": 0
    },
    {
        "path": "new_en.csv",
        "sep": ",",
        "tweet_column": "text",
        "toxicity_columns": [
            "Hate_Speech",
            "Targeted_Harassment",
            "NSFW_Sexual",
            "Violence_Incitement",
            "Dangerous_Ideology",
            "Profanity_Slang"
        ],
        "header": 0
    },
    {
        "path": "rabakbench_en.csv",
        "sep": ",",
        "tweet_column": "text",
        "toxicity_columns": [
            "binary",
            "hateful",
            "insults",
            "sexual",
            "physical_violence",
            "self_harm",
            "all_other_misconduct"
        ],
        "header": 0
    },
    {
        "path": "rabakbench_ta.csv",
        "sep": ",",
        "tweet_column": "text",
        "toxicity_columns": [
            "binary",
            "hateful",
            "insults",
            "sexual",
            "physical_violence",
            "self_harm",
            "all_other_misconduct"
        ],
        "header": 0
    },
    {
        "path": "rabakbench_ms.csv",
        "sep": ",",
        "tweet_column": "text",
        "toxicity_columns": [
            "binary",
            "hateful",
            "insults",
            "sexual",
            "physical_violence",
            "self_harm",
            "all_other_misconduct"
        ],
        "header": 0
    },
    {
        "path": "rabakbench_zh.csv",
        "sep": ",",
        "tweet_column": "text",
        "toxicity_columns": [
            "binary",
            "hateful",
            "insults",
            "sexual",
            "physical_violence",
            "self_harm",
            "all_other_misconduct"
        ],
        "header": 0
    },

    # 示例2：单列字符标签 + 映射
    # {
    #     "path": "English.csv",
    #     "sep": ",",
    #     "tweet_column": "text",
    #     "label_column": "sentiment",
    #     "label_mapping": {
    #         "hate": 1,
    #         "non-hate": 0,
    #         "offensive": 1,
    #         "abusive": 1,
    #         "normal": 0
    #     },
    #     "header": 0
    # },
    # # 示例3：单列已经是0/1（无需映射）
    {
        "path": "ta.csv",
        "sep": ",",
        "tweet_column": "text",
        "label_column": "label",
        "header": 0
    },
    # 更多文件...
]

# ========== 工具函数 ==========
def read_file_with_fallback(file_config):
    """多编码读取CSV/TXT"""
    file_path = file_config["path"]
    sep = file_config.get("sep", ",")
    header = file_config.get("header", 0)
    encoding_list = file_config.get("encoding_try", ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252'])
    for enc in encoding_list:
        try:
            df = pd.read_csv(file_path, sep=sep, header=header, encoding=enc)
            print(f"  ✓ 使用编码 {enc} 成功读取 {file_path}")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"  ✗ 使用编码 {enc} 读取失败: {e}")
            continue
    print(f"  ⚠ 所有编码均失败，使用 latin-1 强制读取")
    return pd.read_csv(file_path, sep=sep, header=header, encoding='latin-1')

def extract_toxicity_label(df, config):
    """
    根据配置返回一个Series（二值标签：0或1），长度与df相同。
    支持两种模式（二选一）：
      1) toxicity_columns: 多列0/1，任意为1则结果为1，否则0。
      2) label_column: 单列标签，若提供label_mapping则映射，否则尝试转换为int(0/1)。
    若两者都无，返回全-1并警告。
    """
    # 模式1：多列二值合并
    if "toxicity_columns" in config:
        cols = config["toxicity_columns"]
        existing = [c for c in cols if c in df.columns]
        if not existing:
            print(f"  ⚠ 未找到任何 toxicity_columns 列: {cols}")
            return pd.Series([-1] * len(df))
        # 按行取最大值（因为0/1，max即OR）
        label_series = df[existing].max(axis=1).fillna(0).astype(int)
        print(f"  ✓ 使用多列合并 (OR) 生成毒性标签，涉及列: {existing}")
        return label_series

    # 模式2：单列标签（可能带映射）
    if "label_column" in config:
        col = config["label_column"]
        if col not in df.columns:
            print(f"  ✗ 未找到 label_column: {col}")
            return pd.Series([-1] * len(df))

        raw = df[col]
        mapping = config.get("label_mapping")

        if mapping is not None:
            # 使用映射字典转换为0/1
            label_series = raw.map(mapping)
            if label_series.isna().any():
                unmapped = raw[label_series.isna()].unique()
                print(f"  ⚠ 发现未映射的值: {unmapped}，将填充为-1")
                label_series = label_series.fillna(-1).astype(int)
            else:
                label_series = label_series.astype(int)
            print(f"  ✓ 使用映射字典转换标签列 '{col}'")
            return label_series
        else:
            # 无映射：尝试直接转为int，如果失败则报错并返回-1
            try:
                label_series = raw.astype(int)
                if label_series.isin([0,1]).all():
                    print(f"  ✓ 标签列 '{col}' 已是0/1整数，直接使用")
                else:
                    print(f"  ⚠ 标签列 '{col}' 包含非0/1的整数，将保留原值（可能不是0/1）")
                return label_series
            except (ValueError, TypeError):
                print(f"  ✗ 标签列 '{col}' 无法转换为整数，且未提供映射 → 标签全为-1")
                return pd.Series([-1] * len(df))

    # 两者都没有
    print(f"  ⚠ 配置中既没有 toxicity_columns 也没有 label_column → 标签全为-1")
    return pd.Series([-1] * len(df))

def extract_tweets_and_labels(df, config):
    """提取推文和标签，返回(tweets列表, labels列表)"""
    tweet_col = config["tweet_column"]
    if tweet_col not in df.columns:
        print(f"  ✗ 未找到推文列 '{tweet_col}'，实际列名: {df.columns.tolist()}")
        return [], []

    # 删除推文为空的行
    non_empty = df[tweet_col].notna()
    valid_df = df[non_empty].copy()

    tweets = valid_df[tweet_col].tolist()

    # 先生成全量标签（长度与原始df相同），再按 non_empty 筛选
    full_labels = extract_toxicity_label(df, config)
    labels = full_labels[non_empty].tolist()
    # 把可能出现的 -1 或其它整数转为字符串（为了统一保存格式）
    labels = [str(l) for l in labels]

    print(f"  提取到 {len(tweets)} 条推文，其中有效标签（0/1）数: {sum(1 for l in labels if l in ('0','1'))}")
    return tweets, labels

# ========== 主函数 ==========
def main():
    all_tweets = []
    all_labels = []

    for idx, config in enumerate(file_configs, 1):
        print(f"\n处理文件 {idx}: {config['path']}")
        df = read_file_with_fallback(config)
        tweets, labels = extract_tweets_and_labels(df, config)
        all_tweets.extend(tweets)
        all_labels.extend(labels)

    # 原始合并文件（去重前）
    raw_df = pd.DataFrame({'tweet': all_tweets, 'label': all_labels})
    raw_output = "all_datasets_with_labels.csv"
    raw_df.to_csv(raw_output, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
    print(f"\n✓ 原始合并: {len(raw_df)} 条记录，已保存至 {raw_output}")

    # 去重（基于 tweet）
    dedup_df = raw_df.drop_duplicates(subset=['tweet'], keep=DEDUP_KEEP)
    removed = len(raw_df) - len(dedup_df)
    print(f"✓ 去重后: {len(dedup_df)} 条记录（移除 {removed} 条重复）")

    dedup_output = "all_datasets_with_labels_dedup.csv"
    dedup_df.to_csv(dedup_output, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
    print(f"✓ 去重结果已保存至 {dedup_output}")

    print("\n去重后前5条预览：")
    print(dedup_df.head())

if __name__ == "__main__":
    main()


处理文件 1: English.csv
  ✓ 使用编码 utf-8 成功读取 English.csv
  ✓ 使用多列合并 (OR) 生成毒性标签，涉及列: ['Hate_Speech', 'Targeted_Harassment', 'NSFW_Sexual', 'Violence_Incitement', 'Dangerous_Ideology', 'Profanity_Slang']
  提取到 44986 条推文，其中有效标签（0/1）数: 44986

处理文件 2: new_en.csv
  ✓ 使用编码 utf-8 成功读取 new_en.csv
  ✓ 使用多列合并 (OR) 生成毒性标签，涉及列: ['Hate_Speech', 'Targeted_Harassment', 'NSFW_Sexual', 'Violence_Incitement', 'Dangerous_Ideology', 'Profanity_Slang']
  提取到 45008 条推文，其中有效标签（0/1）数: 45008

处理文件 3: ta.csv
  ✓ 使用编码 utf-8 成功读取 ta.csv
  ✓ 标签列 'label' 已是0/1整数，直接使用
  提取到 13 条推文，其中有效标签（0/1）数: 13

✓ 原始合并: 90007 条记录，已保存至 all_datasets_with_labels.csv
✓ 去重后: 84472 条记录（移除 5535 条重复）
✓ 去重结果已保存至 all_datasets_with_labels_dedup.csv

去重后前5条预览：
                                               tweet label
0  @USER Ine I put my own bottle and they know be...     0
1                                   @USER you’re old     1
2   @USER @USER @USER Thank you very much Frigid 🙏🏻🥰     0
3  @USER Ask me after the midterms. As it stands ...  

In [12]:
import pandas as pd
import csv

# ========== 去重配置 ==========
DEDUP_KEEP = 'first'   # 'first' 或 'last'

# ========== 文件配置列表 ==========
file_configs = [
    # 示例：单列标签，除了 "normal" 外都是有毒
    {
        "path": "ar_dataset.csv",
        "sep": ",",
        "tweet_column": "tweet",            # 推文所在列
        "label_column": "sentiment",           # 标签列名（包含 normal, hate, offensive 等）
        "label_normal_value": "normal",    # 代表无毒的标签值，其余全为有毒(1)
        "header": 0
    },
    {
        "path": "let-mi_train_part.csv",
        "sep": ",",
        "tweet_column": "text",            # 推文所在列
        "label_column": "category",           # 标签列名（包含 normal, hate, offensive 等）
        "label_normal_value": "none",    # 代表无毒的标签值，其余全为有毒(1)
        "header": 0
    },
        {
        "path": "L-HSAB.csv",
        "sep": ",",
        "tweet_column": "Tweet",
        "label_column": "Class",
        "header": 0
    },
    {
        "path": "offenseval-ar-training-v1.tsv",
        "sep": "\t",
        "tweet_column": "tweet",
        "label_column": "subtask_a",
        "label_normal_value": "NOT",    # 代表无毒的标签值，其余全为有毒(1)
        "header": 0
    },
    # 可以继续添加其他文件（支持多种模式混合）
    # {
    #     "path": "multi_label_file.csv",
    #     "tweet_column": "Tweet",
    #     "toxicity_columns": ["Hate_Speech", "Targeted_Harassment"],
    #     "header": 0
    # },
]

# ========== 工具函数 ==========
def read_file_with_fallback(file_config):
    """多编码读取CSV/TXT"""
    file_path = file_config["path"]
    sep = file_config.get("sep", ",")
    header = file_config.get("header", 0)
    encoding_list = file_config.get("encoding_try", ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252'])
    for enc in encoding_list:
        try:
            df = pd.read_csv(file_path, sep=sep, header=header, encoding=enc)
            print(f"  ✓ 使用编码 {enc} 成功读取 {file_path}")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"  ✗ 使用编码 {enc} 读取失败: {e}")
            continue
    print(f"  ⚠ 所有编码均失败，使用 latin-1 强制读取")
    return pd.read_csv(file_path, sep=sep, header=header, encoding='latin-1')

def extract_toxicity_label(df, config):
    """返回二值标签 Series (0/1/-1)"""
    # 模式1：多列二值合并
    if "toxicity_columns" in config:
        cols = config["toxicity_columns"]
        existing = [c for c in cols if c in df.columns]
        if not existing:
            print(f"  ⚠ 未找到任何 toxicity_columns 列: {cols}")
            return pd.Series([-1] * len(df))
        label_series = df[existing].max(axis=1).fillna(0).astype(int)
        print(f"  ✓ 使用多列合并 (OR) 生成毒性标签，涉及列: {existing}")
        return label_series

    # 模式2：单列标签
    if "label_column" in config:
        col = config["label_column"]
        if col not in df.columns:
            print(f"  ✗ 未找到 label_column: {col}")
            return pd.Series([-1] * len(df))

        raw = df[col]

        # 新增：使用 label_normal_value（除了指定值，其他全为1）
        if "label_normal_value" in config:
            normal_vals = config["label_normal_value"]
            if isinstance(normal_vals, str):
                normal_vals = [normal_vals]
            # 默认全为1（有毒）
            label_series = pd.Series(1, index=df.index)
            # 正常值的位置设为0
            mask_normal = raw.isin(normal_vals)
            label_series[mask_normal] = 0
            # NaN 不会被 isin 匹配，保持为1
            print(f"  ✓ 使用 label_normal_value={normal_vals} 转换：这些值 -> 0，其他所有值（包括缺失）-> 1")
            return label_series

        # 使用映射字典
        mapping = config.get("label_mapping")
        if mapping is not None:
            label_series = raw.map(mapping)
            if label_series.isna().any():
                unmapped = raw[label_series.isna()].unique()
                print(f"  ⚠ 发现未映射的值: {unmapped}，将填充为-1")
                label_series = label_series.fillna(-1).astype(int)
            else:
                label_series = label_series.astype(int)
            print(f"  ✓ 使用映射字典转换标签列 '{col}'")
            return label_series
        else:
            # 无映射：尝试直接转为int
            try:
                label_series = raw.astype(int)
                if label_series.isin([0,1]).all():
                    print(f"  ✓ 标签列 '{col}' 已是0/1整数，直接使用")
                else:
                    print(f"  ⚠ 标签列 '{col}' 包含非0/1的整数，将保留原值")
                return label_series
            except (ValueError, TypeError):
                print(f"  ✗ 标签列 '{col}' 无法转换为整数，且未提供映射或 normal_value → 标签全为-1")
                return pd.Series([-1] * len(df))

    # 两者都没有
    print(f"  ⚠ 配置中既没有 toxicity_columns 也没有 label_column → 标签全为-1")
    return pd.Series([-1] * len(df))

def extract_tweets_and_labels(df, config):
    """提取推文和标签，返回(tweets列表, labels列表)"""
    tweet_col = config["tweet_column"]
    if tweet_col not in df.columns:
        print(f"  ✗ 未找到推文列 '{tweet_col}'，实际列名: {df.columns.tolist()}")
        return [], []

    non_empty = df[tweet_col].notna()
    valid_df = df[non_empty].copy()
    tweets = valid_df[tweet_col].tolist()

    full_labels = extract_toxicity_label(df, config)
    labels = full_labels[non_empty].tolist()
    labels = [str(l) for l in labels]   # 转为字符串保存

    valid_01_count = sum(1 for l in labels if l in ('0','1'))
    print(f"  提取到 {len(tweets)} 条推文，其中有效标签（0/1）数: {valid_01_count}")
    return tweets, labels

def main():
    all_tweets = []
    all_labels = []

    for idx, config in enumerate(file_configs, 1):
        print(f"\n处理文件 {idx}: {config['path']}")
        df = read_file_with_fallback(config)
        tweets, labels = extract_tweets_and_labels(df, config)
        all_tweets.extend(tweets)
        all_labels.extend(labels)

    raw_df = pd.DataFrame({'tweet': all_tweets, 'label': all_labels})
    raw_output = "all_datasets_with_labels.csv"
    raw_df.to_csv(raw_output, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
    print(f"\n✓ 原始合并: {len(raw_df)} 条记录，已保存至 {raw_output}")

    dedup_df = raw_df.drop_duplicates(subset=['tweet'], keep=DEDUP_KEEP)
    removed = len(raw_df) - len(dedup_df)
    print(f"✓ 去重后: {len(dedup_df)} 条记录（移除 {removed} 条重复）")

    dedup_output = "all_datasets_with_labels_dedup.csv"
    dedup_df.to_csv(dedup_output, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
    print(f"✓ 去重结果已保存至 {dedup_output}")

    print("\n去重后前5条预览：")
    print(dedup_df.head())

if __name__ == "__main__":
    main()


处理文件 1: ar_dataset.csv
  ✓ 使用编码 utf-8 成功读取 ar_dataset.csv
  ✓ 使用 label_normal_value=['normal'] 转换：这些值 -> 0，其他所有值（包括缺失）-> 1
  提取到 3353 条推文，其中有效标签（0/1）数: 3353

处理文件 2: let-mi_train_part.csv
  ✓ 使用编码 utf-8 成功读取 let-mi_train_part.csv
  ✓ 使用 label_normal_value=['none'] 转换：这些值 -> 0，其他所有值（包括缺失）-> 1
  提取到 5240 条推文，其中有效标签（0/1）数: 5240

处理文件 3: L-HSAB.csv
  ✓ 使用编码 utf-8 成功读取 L-HSAB.csv
  ✓ 标签列 'Class' 已是0/1整数，直接使用
  提取到 5846 条推文，其中有效标签（0/1）数: 5846

处理文件 4: offenseval-ar-training-v1.tsv
  ✓ 使用编码 utf-8 成功读取 offenseval-ar-training-v1.tsv
  ✓ 使用 label_normal_value=['NOT'] 转换：这些值 -> 0，其他所有值（包括缺失）-> 1
  提取到 7839 条推文，其中有效标签（0/1）数: 7839

✓ 原始合并: 22278 条记录，已保存至 all_datasets_with_labels.csv
✓ 去重后: 22150 条记录（移除 128 条重复）
✓ 去重结果已保存至 all_datasets_with_labels_dedup.csv

去重后前5条预览：
                                               tweet label
0  صلاة الفجر خير لك من ترديد بول البعير وسبي الن...     1
1  صراحة نفسي اشوف ولاد الوسخة اللي قالوا مدرب اج...     1
2  طيب! هي متبرجة وعبايتها ملونه وطالعة من بيتهم ...     

In [3]:
import pandas as pd

# 配置参数
input_file = "ar_dataset.csv"       # 输入 CSV 文件路径
column_name = "sentiment"              # 要提取的列名
output_file = "unique_column_values.csv"  # 输出文件（可选，若不需要可注释）

# 读取 CSV
df = pd.read_csv(input_file, encoding='utf-8')

# 检查列是否存在
if column_name not in df.columns:
    print(f"列 '{column_name}' 不存在。可用列：{df.columns.tolist()}")
else:
    # 提取该列并去重（自动忽略 NaN）
    unique_values = df[column_name].dropna().unique()
    print(f"原数据行数：{len(df)}，该列去重后共 {len(unique_values)} 个唯一值。")
    
    # 打印前20个唯一值（可选）
    print("\n前20个唯一值：")
    for val in unique_values[:30]:
        print(val)
    
    # 保存到新的 CSV 文件（每个唯一值占一行）
    # output_df = pd.DataFrame({column_name: unique_values})
    # output_df.to_csv(output_file, index=False, encoding='utf-8')
    # print(f"\n唯一值已保存至 {output_file}")

原数据行数：3353，该列去重后共 37 个唯一值。

前20个唯一值：
hateful_normal
offensive
normal
offensive_disrespectful
offensive_normal
hateful
abusive_disrespectful
fearful
abusive_hateful
disrespectful
abusive
disrespectful_normal
abusive_offensive
abusive_normal
offensive_hateful
abusive_offensive_hateful_disrespectful_normal
fearful_disrespectful_hateful_normal
abusive_offensive_disrespectful_hateful_normal
fearful_abusive_offensive_hateful_disrespectful
fearful_abusive_disrespectful_hateful_normal
fearful_abusive_offensive_disrespectful_normal
hateful_disrespectful
abusive_offensive_hateful_normal
fearful_offensive_disrespectful_hateful_normal
abusive_offensive_hateful_disrespectful
fearful_abusive_hateful_disrespectful_normal
fearful_normal
fearful_offensive_hateful_normal
abusive_disrespectful_hateful_normal
abusive_offensive_disrespectful_normal
